In [1]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, label_binarize
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import (
    roc_auc_score,
    f1_score,
    confusion_matrix,
    classification_report,
    roc_curve
)

from imblearn.over_sampling import SMOTE


## 1. Cargar datos

In [2]:
url = "https://raw.githubusercontent.com/rosinni/k-nearest-neighbors-project-tutorial/refs/heads/main/winequality-red.csv"
df = pd.read_csv(url, sep=";")
df.head()


,fixed acidity,volatile acidity,citric acid,residual sugar,chlorides,free sulfur dioxide,total sulfur dioxide,density,pH,sulphates,alcohol,quality
0,7.4,0.70,0.00,1.9,0.076,11.0,34.0,0.9978,3.51,0.56,9.4,5
1,7.8,0.88,0.00,2.6,0.098,25.0,67.0,0.9968,3.20,0.68,9.8,5
2,7.8,0.76,0.04,2.3,0.092,15.0,54.0,0.9970,3.26,0.65,9.8,5
3,11.2,0.28,0.56,1.9,0.075,17.0,60.0,0.9980,3.16,0.58,9.8,6
4,7.4,0.70,0.00,1.9,0.076,11.0,34.0,0.9978,3.51,0.56,9.4,5


In [3]:
df.quality.value_counts()

quality
5    681
6    638
7    199
4     53
8     18
3     10
Name: count, dtype: int64

## 2. Crear las 3 clases

Creamos una funcion que nos agrupe los datos segun la siguiente regla:

- `quality <= 4`  → `0` = Baja
- `quality 5 o 6` → `1` = Media
- `quality >= 7`  → `2` = Alta


In [4]:
def quality_to_label(q):
    if q <= 4:
        return 0   # baja
    elif q <= 6:
        return 1   # media
    else:
        return 2   # alta

df["label"] = df["quality"].apply(quality_to_label)
df[["quality", "label"]].head(10)


,quality,label
0,5,1
1,5,1
2,5,1
3,6,1
4,5,1
5,5,1
6,5,1
7,7,2
8,7,2
9,5,1


## 3. Ver distribución original de clases

In [5]:
print("Distribución original:")
print(df["label"].value_counts().sort_index())


Distribución original:
label
0      63
1    1319
2     217
Name: count, dtype: int64


In [6]:
# Graficamos la distribucion de las clases con plotly
conteo = df["label"].value_counts().sort_index().reset_index()
conteo.columns = ["label", "cantidad"]
conteo["grupo"] = conteo["label"].map({0: "Baja", 1: "Media", 2: "Alta"})

fig = px.bar(
    conteo,
    x="grupo",
    y="cantidad",
    text="cantidad",
    title="Distribución original de clases"
)
fig.show()


## 4. Separar variables

- `X` = variables de entrada
- `y` = clase objetivo


In [7]:
# Sacamos las variables "quality" y "label" de X, ya que, "label" contiene a "quality" y "label" es la variable objetivo.
X = df.drop(columns=["quality", "label"])
y = df["label"]

print("Número de variables:", X.shape[1])
print("Columnas:")
print(list(X.columns))


Número de variables: 11
Columnas:
['fixed acidity', 'volatile acidity', 'citric acid', 'residual sugar', 'chlorides', 'free sulfur dioxide', 'total sulfur dioxide', 'density', 'pH', 'sulphates', 'alcohol']


## 5. Separar train y test

Usamos `stratify=y` para mantener la proporción de clases.


In [8]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Tamaño train:", X_train.shape)
print("Tamaño test :", X_test.shape)


Tamaño train: (1279, 11)
Tamaño test : (320, 11)


## 6. Escalar los datos

Esto es importante porque KNN usa distancias.


In [9]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)


## 7. Aplicar SMOTE solo al train

Esto genera ejemplos sintéticos de la clase minoritaria para balancear el entrenamiento.


In [10]:
smote = SMOTE(random_state=42, k_neighbors=3)
X_train_bal, y_train_bal = smote.fit_resample(X_train_scaled, y_train)

print("Distribución después de SMOTE (solo train):")
print(pd.Series(y_train_bal).value_counts().sort_index())


Distribución después de SMOTE (solo train):
label
0    1055
1    1055
2    1055
Name: count, dtype: int64


In [11]:
# Volvemos a graficar la distribucion
conteo_bal = pd.Series(y_train_bal).value_counts().sort_index().reset_index()
conteo_bal.columns = ["label", "cantidad"]
conteo_bal["grupo"] = conteo_bal["label"].map({0: "Baja", 1: "Media", 2: "Alta"})

fig = px.bar(
    conteo_bal,
    x="grupo",
    y="cantidad",
    text="cantidad",
    title="Distribución de clases en train después de SMOTE"
)
fig.show()


## 8. Buscar el mejor `k`

Probamos distintos valores de `k` y guardamos:

- **AUC multiclase**
- **F1 macro**
- **F1 weighted**


In [12]:
# Iteramos el entrenamiento y test del modelo con diferentes valores de K y obtenemos los resultados en un Data Frame
k_range = range(1, 31)
resultados = []

for k in k_range:
    model = KNeighborsClassifier(
        n_neighbors=k,
        weights="distance"
    )
    model.fit(X_train_bal, y_train_bal)

    y_pred = model.predict(X_test_scaled)
    y_proba = model.predict_proba(X_test_scaled)

    auc = roc_auc_score(y_test, y_proba, multi_class="ovr")
    f1_macro = f1_score(y_test, y_pred, average="macro")
    f1_weighted = f1_score(y_test, y_pred, average="weighted")

    resultados.append({
        "k": k,
        "auc": auc,
        "f1_macro": f1_macro,
        "f1_weighted": f1_weighted
    })

resultados_k = pd.DataFrame(resultados)
resultados_k

,k,auc,f1_macro,f1_weighted
0,1,0.707901,0.571787,0.813586
1,2,0.787954,0.571787,0.813586
2,3,0.781815,0.595130,0.807074
3,4,0.781055,0.591662,0.804840
4,5,0.786961,0.572614,0.786543
5,6,0.792704,0.578079,0.791471
6,7,0.790998,0.522255,0.737765
7,8,0.806452,0.536159,0.752945
8,9,0.807979,0.513498,0.727965
9,10,0.805188,0.512876,0.727027


## 9. Gráfico: AUC multiclase según `k`

In [13]:
# Graficamos el AUC para los distintos valores de k
fig = px.line(
    resultados_k,
    x="k",
    y="auc",
    markers=True,
    title="AUC multiclase según k (SMOTE + KNN)"
)
fig.show()


## 10. Gráfico: F1 macro según `k`

In [14]:
# Graficamos el F1 macro para los distintos valores de k
fig = px.line(
    resultados_k,
    x="k",
    y="f1_macro",
    markers=True,
    title="F1 macro según k (SMOTE + KNN)"
)
fig.show()


## 11. Gráfico: F1 weighted según `k`

In [15]:
# Graficamos el F1 weighted para los distintos valores de k
fig = px.line(
    resultados_k,
    x="k",
    y="f1_weighted",
    markers=True,
    title="F1 weighted según k (SMOTE + KNN)"
)
fig.show()


## 12. Elegir el mejor `k`

Aquí elegimos el mejor `k` usando **F1 macro**, porque queremos dar el mismo peso a las 3 clases.


In [16]:
best_k = int(resultados_k.loc[resultados_k["f1_macro"].idxmax(), "k"])

print("Mejor k según F1 macro:", best_k)


Mejor k según F1 macro: 3


In [17]:
best_k

3

## 13. Entrenar el modelo final

In [18]:
# Entrenamiento y prueba del modelo con el mejor k
final_knn = KNeighborsClassifier(
    n_neighbors=best_k,
    weights="distance"
)
final_knn.fit(X_train_bal, y_train_bal)

y_pred_final = final_knn.predict(X_test_scaled)
y_proba_final = final_knn.predict_proba(X_test_scaled)

final_auc = roc_auc_score(y_test, y_proba_final, multi_class="ovr")
final_f1_macro = f1_score(y_test, y_pred_final, average="macro")
final_f1_weighted = f1_score(y_test, y_pred_final, average="weighted")

print("=== Modelo final con SMOTE ===")
print(f"AUC final: {final_auc:.4f}")
print(f"F1 macro final: {final_f1_macro:.4f}")
print(f"F1 weighted final: {final_f1_weighted:.4f}")


=== Modelo final con SMOTE ===
AUC final: 0.7818
F1 macro final: 0.5951
F1 weighted final: 0.8071


## 14. Reporte de clasificación

Usamos `zero_division=0` para evitar warnings si alguna clase no se predice.


In [19]:
print(classification_report(
    y_test,
    y_pred_final,
    target_names=["Baja", "Media", "Alta"],
    zero_division=0
))


              precision    recall  f1-score   support

        Baja       0.24      0.46      0.32        13
       Media       0.93      0.81      0.86       264
        Alta       0.50      0.77      0.61        43

    accuracy                           0.79       320
   macro avg       0.56      0.68      0.60       320
weighted avg       0.84      0.79      0.81       320



## 15. Matriz de confusión final

In [20]:
cm_final = confusion_matrix(y_test, y_pred_final)

fig = px.imshow(
    cm_final,
    text_auto=True,
    x=["Pred Baja", "Pred Media", "Pred Alta"],
    y=["Real Baja", "Real Media", "Real Alta"],
    title="Matriz de confusión final (SMOTE + KNN)"
)
fig.show()


## 16. Curvas ROC por clase

En multiclase usamos el enfoque **One-vs-Rest**.


In [21]:
classes = [0, 1, 2]
class_names = {0: "Baja", 1: "Media", 2: "Alta"}

y_test_bin = label_binarize(y_test, classes=classes)

fig = go.Figure()

for i, cls in enumerate(classes):
    fpr, tpr, _ = roc_curve(y_test_bin[:, i], y_proba_final[:, i])
    auc_cls = roc_auc_score(y_test_bin[:, i], y_proba_final[:, i])

    fig.add_trace(go.Scatter(
        x=fpr,
        y=tpr,
        mode="lines",
        name=f"{class_names[cls]} (AUC={auc_cls:.3f})"
    ))

fig.add_trace(go.Scatter(
    x=[0, 1],
    y=[0, 1],
    mode="lines",
    name="Línea base",
    line=dict(dash="dash")
))

fig.update_layout(
    title="Curvas ROC por clase",
    xaxis_title="False Positive Rate",
    yaxis_title="True Positive Rate"
)
fig.show()


## 17. Scatter para entender mejor los datos

No usa todas las variables, pero ayuda a visualizar el problema.


In [22]:
df_plot = df.copy()
df_plot["grupo"] = df_plot["label"].map({0: "Baja", 1: "Media", 2: "Alta"})

fig = px.scatter(
    df_plot,
    x="alcohol",
    y="volatile acidity",
    color="grupo",
    title="Vista simple de los vinos por alcohol y acidez volátil",
    hover_data=["quality"]
)
fig.show()


## 18. Probabilidades predichas por clase

Este gráfico ayuda a ver cómo reparte el modelo sus probabilidades.


In [23]:
probas_df = pd.DataFrame(
    y_proba_final,
    columns=["Prob Baja", "Prob Media", "Prob Alta"]
)
probas_df["Clase real"] = y_test.reset_index(drop=True).map({
    0: "Baja",
    1: "Media",
    2: "Alta"
})

probas_largas = probas_df.melt(
    id_vars="Clase real",
    var_name="Clase probabilidad",
    value_name="Probabilidad"
)

fig = px.box(
    probas_largas,
    x="Clase probabilidad",
    y="Probabilidad",
    color="Clase real",
    title="Distribución de probabilidades predichas"
)
fig.show()


## 19. Función de predicción

Devuelve:

- clase predicha
- probabilidad de cada clase


In [24]:
def predict_wine_quality(features):
    nuevo_vino = pd.DataFrame([features], columns=X.columns)
    nuevo_vino_scaled = scaler.transform(nuevo_vino)

    pred = final_knn.predict(nuevo_vino_scaled)[0]
    proba = final_knn.predict_proba(nuevo_vino_scaled)[0]

    etiquetas = {
        0: "Calidad baja 🍷",
        1: "Calidad media 🍷",
        2: "Calidad alta 🍷"
    }

    return {
        "prediccion": etiquetas[pred],
        "prob_baja": round(float(proba[0]), 4),
        "prob_media": round(float(proba[1]), 4),
        "prob_alta": round(float(proba[2]), 4)
    }


## 20. Probar la función

In [25]:
ejemplo = {
    "fixed acidity": 7.4,
    "volatile acidity": 0.7,
    "citric acid": 0.0,
    "residual sugar": 1.9,
    "chlorides": 0.076,
    "free sulfur dioxide": 11.0,
    "total sulfur dioxide": 34.0,
    "density": 0.9978,
    "pH": 3.51,
    "sulphates": 0.56,
    "alcohol": 9.4
}

print("Predicción ejemplo:")
print(predict_wine_quality(ejemplo))


Predicción ejemplo:
{'prediccion': 'Calidad media 🍷', 'prob_baja': 0.0, 'prob_media': 1.0, 'prob_alta': 0.0}
